# Notebook 04 — Biological Reasoning Engine

**Purpose**: Take a `DeviationReport` (the structured output of the Healthy Respiratory Model)
and produce a rich, human-readable biological interpretation using chain-of-thought reasoning.

The engine explains **why** abnormalities matter, reasons about underlying biological
mechanisms, and draws logical implications — without naming diseases or making clinical diagnoses.

**Pipeline**:
```
DeviationReport  →  Evidence Triage  →  Pattern Clustering  →  Context Enrichment
                 →  Chain-of-Thought Prompt  →  Claude API  →  BiologicalInterpretation
```

**Prerequisites**: Run Notebook 02 first. Download artefacts to Drive before running this.

---
**SAFETY NOTICE**: All outputs from this engine are research tools intended for qualified
scientists and clinicians. They do not constitute clinical diagnoses. Every interpretation
requires validation by certified medical professionals.
---

---
## Section 1 — Setup

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('anthropic>=0.28.0')
print('Dependencies ready.')

In [ ]:
import json
import os
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import anthropic

print('Imports OK.')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# ── Paths — edit if your Drive layout differs ─────────────────────────────────
MODEL_DIR = Path('/content/drive/MyDrive/HEALTH/models/respiratory_model')

# ── API key — stored as a Colab secret named ANTHROPIC_API_KEY ────────────────
# Add via: Colab → Secrets (key icon) → New secret → Name: ANTHROPIC_API_KEY
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

assert MODEL_DIR.exists(), f'Model artefacts not found at {MODEL_DIR}. Run Notebook 02 first.'
assert ANTHROPIC_API_KEY,  'Set ANTHROPIC_API_KEY in Colab Secrets.'

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print(f'Model artefacts: {MODEL_DIR}')
print('Anthropic client ready.')

---
## Section 2 — Load Artefacts

In [ ]:
def load_json(name: str) -> dict:
    p = MODEL_DIR / name
    if not p.exists():
        return {}
    return json.loads(p.read_text())


composition       = load_json('cell_type_composition.json')
gene_stats        = load_json('gene_stats.json')
cell_type_profiles = load_json('cell_type_profiles.json')
pathway_baselines = load_json('pathway_baselines.json')
fetal_reference   = load_json('fetal_reference.json')
spatial_baselines = load_json('spatial_baselines.json')
meta              = load_json('meta.json')

print(f'Healthy reference built from {meta.get("n_cells", "?"):,} cells across {meta.get("n_donors", "?")} donors')
print(f'Cell types:  {len(cell_type_profiles)}')
print(f'Genes:       {len(gene_stats):,}')
print(f'Pathways:    {len(pathway_baselines)}')
print(f'Fetal ref:   {"yes" if fetal_reference.get("fetal_enriched") else "not available"}')
print(f'Spatial:     {"yes" if spatial_baselines.get("bronchus") else "not available"}')

---
## Section 3 — Evidence Triage

Before reasoning, rank and filter findings so the model receives the most signal-rich
evidence first. Weak deviations (|Z| < 2, small pathway deltas) are noted but not
foregrounded — they add noise without proportional insight.

In [ ]:
# ── Severity thresholds (mirror anomaly_detector.py) ─────────────────────────
GENE_Z_MODERATE  = 2.0
GENE_Z_SEVERE    = 4.0
CT_Z_MODERATE    = 2.0
CT_Z_SEVERE      = 3.5
PATHWAY_DELTA    = 0.8   # log1p CP10K units
FETAL_NOTABLE    = 0.15

# How many top findings to send to the reasoning model
MAX_GENE_FINDINGS     = 20
MAX_CT_FINDINGS       = 10
MAX_PATHWAY_FINDINGS  = 12


def triage(report: dict) -> dict:
    """
    Sort and filter all deviation dimensions by severity.
    Returns a dict of ranked findings ready for pattern clustering.
    """
    # Gene deviations — sort by |z_score| descending
    gene_devs = sorted(
        report.get('gene_deviations', []),
        key=lambda x: abs(x.get('z_score', 0)),
        reverse=True,
    )[:MAX_GENE_FINDINGS]

    # Cell type deviations — sort by |z_score| descending
    ct_devs = sorted(
        report.get('cell_type_deviations', []),
        key=lambda x: abs(x.get('z_score', 0)),
        reverse=True,
    )[:MAX_CT_FINDINGS]

    # Pathway deviations — sort by |deviation_from_baseline| then avg_expression
    pw_devs = sorted(
        report.get('pathway_deviations', []),
        key=lambda x: abs(x.get('deviation_from_baseline') or x.get('avg_expression', 0)),
        reverse=True,
    )[:MAX_PATHWAY_FINDINGS]

    fetal = report.get('fetal_reactivation')

    return {
        'overall_score':      report.get('overall_deviation_score', 0.0),
        'gene_deviations':    gene_devs,
        'cell_type_deviations': ct_devs,
        'pathway_deviations': pw_devs,
        'fetal_reactivation': fetal,
        'sample_id':          report.get('sample_id', 'unknown'),
        'model_built_at':     report.get('model_built_at', ''),
        'ref_cells':          report.get('healthy_reference_cells', 0),
        'ref_donors':         report.get('healthy_reference_donors', 0),
    }


print('Evidence triage function defined.')

---
## Section 4 — Pattern Clustering

Group findings that point to the same biological process. A depleted AT2 cell population,
suppressed surfactant genes, and a suppressed Surfactant/AT2 pathway are three separate
numbers in the report — but they describe the same phenomenon. Clustering them lets the
reasoning model see the coherent picture rather than isolated data points.

In [ ]:
# Biological groupings: which genes and cell types relate to which process
_PROCESS_MAP = {
    'alveolar_integrity': {
        'genes':      ['SFTPC', 'SFTPB', 'SFTPA1', 'SFTPA2', 'ABCA3', 'LAMP3', 'NKX2-1'],
        'cell_types': ['AT2', 'AT1', 'alveolar type 2', 'alveolar type 1'],
        'pathways':   ['Surfactant / AT2 Function'],
        'label':      'Alveolar integrity',
    },
    'fibrosis_ecm': {
        'genes':      ['COL1A1', 'COL3A1', 'COL5A1', 'FN1', 'TGFB1', 'TGFB2', 'ACTA2', 'VIM', 'POSTN', 'MMP2', 'TIMP1'],
        'cell_types': ['fibroblast', 'myofibroblast', 'smooth muscle'],
        'pathways':   ['TGF-β / Fibrosis'],
        'label':      'Fibrosis / ECM remodelling',
    },
    'immune_activation': {
        'genes':      ['MX1', 'ISG15', 'IFIT1', 'IFIT3', 'OAS1', 'OAS2', 'RSAD2', 'STAT1', 'IRF7', 'CXCL10'],
        'cell_types': ['alveolar macrophage', 'monocyte', 'dendritic cell', 'NK cell', 'CD8+ T cell'],
        'pathways':   ['Interferon Response', 'NF-kB / Cytokine Storm', 'Myeloid Activation'],
        'label':      'Immune activation / inflammatory signalling',
    },
    'oncogenic_reprogramming': {
        'genes':      ['MKI67', 'TOP2A', 'PCNA', 'CCND1', 'CCNE1', 'CDK4', 'RB1', 'TP53', 'CDKN2A', 'AURKB'],
        'cell_types': [],
        'pathways':   ['Cell Proliferation / Oncogenic'],
        'label':      'Aberrant proliferation / cell cycle dysregulation',
    },
    'mucociliary': {
        'genes':      ['FOXJ1', 'DNAI1', 'DNAI2', 'MUC5AC', 'MUC5B', 'SCGB1A1', 'CFTR'],
        'cell_types': ['multiciliated columnar cell', 'goblet cell', 'club cell', 'basal cell'],
        'pathways':   ['Mucociliary Clearance'],
        'label':      'Airway mucociliary function',
    },
    'vascular': {
        'genes':      ['PECAM1', 'CDH5', 'VWF', 'THBD', 'EPCAM', 'VEGFA', 'ANGPT2'],
        'cell_types': ['vascular endothelial', 'capillary aerocyte', 'pericyte'],
        'pathways':   ['Vascular Injury / Coagulation'],
        'label':      'Vascular / endothelial integrity',
    },
    'hypoxia_stress': {
        'genes':      ['HIF1A', 'VEGFA', 'LDHA', 'SLC2A1', 'EPO', 'ADM', 'BNIP3'],
        'cell_types': [],
        'pathways':   ['Hypoxia / HIF Response'],
        'label':      'Hypoxic stress / metabolic reprogramming',
    },
    'lymphoid_exhaustion': {
        'genes':      ['PDCD1', 'CTLA4', 'HAVCR2', 'LAG3', 'TIGIT', 'TOX', 'PRDM1'],
        'cell_types': ['CD8+ T cell', 'CD4+ T cell', 'regulatory T cell', 'NK cell'],
        'pathways':   ['T Cell Exhaustion', 'Complement Activation'],
        'label':      'T cell dysfunction / lymphoid exhaustion',
    },
    'developmental_reactivation': {
        'genes':      [],   # populated from fetal_reference at runtime
        'cell_types': [],
        'pathways':   [],
        'label':      'Fetal / developmental programme reactivation',
    },
}


def cluster_findings(triaged: dict) -> list[dict]:
    """
    Group triaged findings into coherent biological processes.
    Each cluster contains the evidence that supports it.
    Returns clusters sorted by evidence weight (number of supporting findings).
    """
    active_genes     = {d['gene']:    d for d in triaged['gene_deviations']}
    active_cts       = {d['cell_type']: d for d in triaged['cell_type_deviations']}
    active_pathways  = {d['pathway']:  d for d in triaged['pathway_deviations']}
    fetal            = triaged.get('fetal_reactivation')

    # Augment developmental cluster with actual fetal genes if available
    if fetal_reference.get('fetal_enriched'):
        _PROCESS_MAP['developmental_reactivation']['genes'] = list(
            fetal_reference['fetal_enriched'].keys()
        )[:30]

    clusters = []
    for process_key, spec in _PROCESS_MAP.items():
        supporting_genes    = [g for g in spec['genes']    if g in active_genes]
        supporting_cts      = [ct for ct in spec['cell_types'] if any(ct.lower() in k.lower() for k in active_cts)]
        supporting_pathways = [pw for pw in spec['pathways'] if pw in active_pathways]

        # Skip processes with no supporting evidence
        evidence_count = len(supporting_genes) + len(supporting_cts) + len(supporting_pathways)
        if evidence_count == 0:
            continue

        # Fetal reactivation cluster — also check score
        if process_key == 'developmental_reactivation':
            if not fetal or fetal.get('score', 0) < FETAL_NOTABLE:
                if evidence_count < 2:
                    continue

        gene_evidence = [
            {
                'gene':      g,
                'z_score':   active_genes[g].get('z_score', 0),
                'direction': active_genes[g].get('direction', ''),
                'magnitude': active_genes[g].get('magnitude', ''),
            }
            for g in supporting_genes
        ]
        ct_evidence = [
            {
                'cell_type': k,
                'z_score':   active_cts[k].get('z_score', 0),
                'direction': active_cts[k].get('direction', ''),
            }
            for k in active_cts
            if any(ct.lower() in k.lower() for ct in spec['cell_types'])
        ]
        pw_evidence = [
            {
                'pathway':   pw,
                'direction': active_pathways[pw].get('direction', ''),
                'deviation': active_pathways[pw].get('deviation_from_baseline'),
            }
            for pw in supporting_pathways
        ]

        clusters.append({
            'process':          spec['label'],
            'evidence_count':   evidence_count,
            'gene_evidence':    gene_evidence,
            'ct_evidence':      ct_evidence,
            'pathway_evidence': pw_evidence,
        })

    clusters.sort(key=lambda x: x['evidence_count'], reverse=True)
    return clusters


print('Pattern clustering function defined.')

---
## Section 5 — Context Enrichment

Before prompting, attach biological context to each finding from the model artefacts:
what a gene normally does in healthy lung, what healthy baseline levels are, what
the functional role of each cell type is. This grounds the reasoning in biology
rather than leaving it to infer from gene names alone.

In [ ]:
# Curated functional annotations for key respiratory genes
# These are biological facts, not disease signatures — they describe what each
# component does in a healthy respiratory system.
_GENE_ROLES = {
    'SFTPC':  'Surfactant protein C — secreted by AT2 cells to reduce alveolar surface tension; essential for normal breathing mechanics',
    'SFTPB':  'Surfactant protein B — required for lamellar body packaging and surfactant function; loss causes acute respiratory failure',
    'SFTPA1': 'Surfactant protein A1 — collectin involved in innate immune defence and surfactant homeostasis',
    'ABCA3':  'ATP-binding cassette transporter A3 — packages phospholipids into lamellar bodies in AT2 cells; essential for surfactant biogenesis',
    'NKX2-1': 'Transcription factor (TTF-1) — master regulator of lung identity in AT2 and airway epithelial cells; controls surfactant gene expression',
    'FOXJ1':  'Transcription factor — required for ciliogenesis; its loss indicates failure of multiciliated cell differentiation',
    'MUC5AC': 'Gel-forming mucin secreted by goblet cells — component of the airway mucus layer that traps inhaled particles',
    'MUC5B':  'Gel-forming mucin — major structural component of airway mucus; critical for mucociliary clearance',
    'MARCO':  'Macrophage receptor with collagenous structure — scavenger receptor on alveolar macrophages for inhaled particles and pathogens',
    'MX1':    'MX dynamin-like GTPase 1 — interferon-stimulated antiviral effector; normally silent in healthy lung, strongly induced by viral infection',
    'ISG15':  'Interferon-stimulated gene 15 — ubiquitin-like modifier; antiviral, silent at baseline, activated within hours of type I interferon signalling',
    'IFIT1':  'Interferon-induced protein with tetratricopeptide repeats 1 — strong antiviral response gene, near-zero in healthy tissue',
    'TGFB1':  'Transforming growth factor beta 1 — pleiotropic cytokine; in lung, its sustained elevation drives myofibroblast activation and collagen deposition',
    'COL1A1': 'Collagen type I alpha 1 — fibrillar collagen; in healthy lung, expressed at very low levels; its elevation marks active extracellular matrix remodelling',
    'COL3A1': 'Collagen type III alpha 1 — fibrillar collagen co-expressed with COL1A1 in early fibrotic response',
    'ACTA2':  'Alpha smooth muscle actin — marker of myofibroblast activation; traces of expression are normal, but elevation signals contractile fibroblast activation',
    'MKI67':  'Ki-67 — expressed exclusively in cycling cells; near-zero in healthy lung epithelium (post-mitotic); its elevation marks proliferating cells',
    'TOP2A':  'Topoisomerase II alpha — required for DNA replication; expressed only in S/G2/M phase; elevated alongside MKI67 in proliferating populations',
    'PCNA':   'Proliferating cell nuclear antigen — DNA clamp for replication; another proliferation marker normally absent from quiescent lung cells',
    'TP53':   'Tumour protein p53 — guardian of the genome; its suppression removes the brake on aberrant proliferation',
    'RB1':    'Retinoblastoma protein — cell cycle checkpoint; loss de-represses E2F transcription factors, driving S-phase entry',
    'PECAM1': 'Platelet endothelial cell adhesion molecule (CD31) — expressed on vascular endothelium; marks endothelial integrity',
    'VWF':    'von Willebrand factor — secreted by endothelial cells; stored in Weibel-Palade bodies; elevated in endothelial activation or injury',
    'HIF1A':  'Hypoxia-inducible factor 1 alpha — master transcription factor for hypoxic adaptation; normally kept low by prolyl hydroxylase-mediated degradation',
    'PDCD1':  'Programmed death receptor 1 (PD-1) — co-inhibitory receptor; sustained high expression marks T cell exhaustion',
    'CTLA4':  'Cytotoxic T-lymphocyte antigen 4 — inhibitory checkpoint receptor; its upregulation dampens T cell activation',
    'HAVCR2': 'TIM-3 — exhaustion marker co-expressed with PD-1 in terminally exhausted T cells; also marks macrophage activation states',
    'VEGFA':  'Vascular endothelial growth factor A — primary pro-angiogenic signal; in lung, upregulated by hypoxia and during vascular repair',
    'IL6':    'Interleukin-6 — pleiotropic pro-inflammatory cytokine; near-silent in healthy lung; rapid induction marks acute inflammatory response',
    'CXCL10': 'C-X-C motif chemokine 10 (IP-10) — interferon-gamma-induced; recruits T cells and NK cells to sites of viral infection or inflammation',
    'STAT1':  'Signal transducer and activator of transcription 1 — phosphorylated downstream of interferon receptors; drives ISG expression',
    'SCGB1A1': 'Secretoglobin family 1A member 1 (CC16/CCSP) — secreted by club cells; anti-inflammatory; its loss indicates club cell dysfunction or loss',
    'CFTR':   'Cystic fibrosis transmembrane conductance regulator — chloride channel in airway epithelium; essential for airway surface liquid homeostasis',
    'FN1':    'Fibronectin 1 — extracellular matrix glycoprotein; elevated in active wound healing and fibrotic remodelling',
    'SPP1':   'Osteopontin — secreted phosphoprotein; in macrophages, marks a pro-inflammatory, matrix-remodelling subtype',
}

_CELL_TYPE_ROLES = {
    'AT2':                        'Alveolar type 2 cells — progenitors of the alveolar epithelium; secrete surfactant to maintain alveolar patency; self-renewing',
    'AT1':                        'Alveolar type 1 cells — thin, squamous cells covering ~95% of alveolar surface; primary site of gas exchange',
    'alveolar macrophage':        'Resident alveolar macrophages — first line of innate immune defence; phagocytose debris, pathogens, and surfactant; maintain alveolar homeostasis',
    'multiciliated columnar cell': 'Airway multiciliated cells — beat in coordinated waves to move mucus and trapped particles out of the airway',
    'club cell':                  'Club (Clara) cells — non-ciliated secretory cells; produce CC16 (anti-inflammatory); progenitors for ciliated cells after injury',
    'fibroblast':                 'Lung fibroblasts — stromal cells that maintain the extracellular matrix scaffold; become myofibroblasts during injury/repair',
    'basal cell':                 'Airway basal cells — stem cells of the bronchial epithelium; replenish ciliated and secretory cells after injury',
    'vascular endothelial':       'Vascular endothelial cells — line blood vessels; regulate gas exchange, coagulation, and immune cell trafficking',
    'CD8+ T cell':                'Cytotoxic T lymphocytes — kill virus-infected or transformed cells; recruited from circulation during immune activation',
    'NK cell':                    'Natural killer cells — innate cytotoxic lymphocytes; survey for cells lacking MHC-I (infected or transformed)',
    'monocyte':                   'Monocytes — circulating precursors recruited to lung during inflammation; differentiate into macrophages or dendritic cells',
    'regulatory T cell':          'Regulatory T cells (Tregs) — suppress immune responses; their depletion can permit unchecked inflammation',
    'goblet cell':                'Goblet cells — mucus-secreting epithelial cells; their expansion causes mucus hypersecretion when excessive',
    'smooth muscle cell':         'Airway smooth muscle cells — regulate airway calibre; their contraction causes bronchoconstriction',
}


def enrich_with_context(clusters: list[dict], triaged: dict) -> dict:
    """Add healthy baseline values and functional annotations to each finding."""
    # Enrich gene deviations
    enriched_genes = []
    for dev in triaged['gene_deviations']:
        gene  = dev['gene']
        stats = gene_stats.get(gene, {})
        enriched_genes.append({
            **dev,
            'healthy_mean':  stats.get('mean', None),
            'healthy_p5':    stats.get('p5',   None),
            'healthy_p95':   stats.get('p95',  None),
            'functional_role': _GENE_ROLES.get(gene, f'{gene} — role not annotated in this reference'),
        })

    # Enrich cell type deviations
    enriched_cts = []
    for dev in triaged['cell_type_deviations']:
        ct = dev['cell_type']
        role = next(
            (v for k, v in _CELL_TYPE_ROLES.items() if k.lower() in ct.lower() or ct.lower() in k.lower()),
            f'{ct} — functional role not annotated in this reference',
        )
        enriched_cts.append({**dev, 'functional_role': role})

    fetal = triaged.get('fetal_reactivation')
    if fetal:
        fetal = {
            **fetal,
            'biological_meaning': (
                'Fetal gene programmes are normally silenced in adult lung. '
                'Their re-expression indicates dedifferentiation, developmental progenitor '
                'expansion, or oncogenic reprogramming toward an immature cell state.'
            ),
        }

    return {
        'sample_id':           triaged['sample_id'],
        'overall_score':       triaged['overall_score'],
        'ref_cells':           triaged['ref_cells'],
        'ref_donors':          triaged['ref_donors'],
        'model_built_at':      triaged['model_built_at'],
        'clusters':            clusters,
        'gene_deviations':     enriched_genes,
        'cell_type_deviations': enriched_cts,
        'pathway_deviations':  triaged['pathway_deviations'],
        'fetal_reactivation':  fetal,
    }


print('Context enrichment function defined.')

---
## Section 6 — Chain-of-Thought Reasoning Engine

This is the core of the notebook. The enriched evidence is formatted into a structured
prompt and sent to Claude, which reasons step by step through:

1. **Evidence review** — what the data is actually saying
2. **Pattern recognition** — which findings cluster into coherent biological processes
3. **Mechanism reasoning** — what biological mechanisms could produce this pattern
4. **Functional implications** — what does this mean for lung physiology
5. **Uncertainty and gaps** — what the data cannot tell us
6. **Final interpretation** — the concise biological summary

The model is instructed never to name diseases or diagnose. Its role is to characterise biology.

In [ ]:
SYSTEM_PROMPT = """\
You are a respiratory biology research assistant integrated into the HEALTH platform.

Your role is to interpret gene expression deviation data produced by the Healthy Respiratory
Model — a reference built from 1.2 million healthy human lung cells (Human Lung Cell Atlas).

CRITICAL CONSTRAINTS:
- You MUST NOT name specific diseases or clinical diagnoses. Your output is for research and
  clinical assistance — the naming and diagnosis belongs to certified medical professionals.
- Do not speculate beyond the biological evidence provided.
- When findings are ambiguous or contradictory, say so clearly.
- Every interpretation carries uncertainty. Quantify it where you can.

Your output characterises BIOLOGY: what is happening in this tissue at the cellular and
molecular level, and what that implies for respiratory function. You describe mechanisms.
Clinicians draw conclusions.

Think step by step through the evidence before writing your final interpretation.
"""


def build_reasoning_prompt(enriched: dict) -> str:
    lines = []

    lines.append(f'SAMPLE: {enriched["sample_id"]}')
    lines.append(f'Overall deviation score: {enriched["overall_score"]:.3f} (0=healthy, 1=maximally abnormal)')
    lines.append(f'Healthy reference: {enriched["ref_cells"]:,} cells from {enriched["ref_donors"]} donors')
    lines.append('')

    # Biological clusters
    if enriched['clusters']:
        lines.append('── BIOLOGICAL PROCESS CLUSTERS ─────────────────────────────────────────────')
        for cl in enriched['clusters']:
            lines.append(f'\n[{cl["process"]}] ({cl["evidence_count"]} supporting findings)')
            if cl['gene_evidence']:
                for g in cl['gene_evidence']:
                    lines.append(f'  Gene  {g["gene"]:10s}  Z={g["z_score"]:+.1f}  {g["direction"]}  ({g["magnitude"]})')
            if cl['ct_evidence']:
                for ct in cl['ct_evidence']:
                    lines.append(f'  Cell  {ct["cell_type"]:30s}  Z={ct["z_score"]:+.1f}  {ct["direction"]}')
            if cl['pathway_evidence']:
                for pw in cl['pathway_evidence']:
                    delta = f'{pw["deviation"]:+.2f} log1p' if pw['deviation'] is not None else 'no baseline'
                    lines.append(f'  Pathway  {pw["pathway"]:35s}  {pw["direction"]}  Δ={delta}')
        lines.append('')

    # Top gene deviations with functional context
    if enriched['gene_deviations']:
        lines.append('── TOP GENE DEVIATIONS ─────────────────────────────────────────────────────')
        for d in enriched['gene_deviations'][:12]:
            lines.append(f'  {d["gene"]:10s}  sample={d.get("sample_value", 0):.2f}  '
                         f'healthy_mean={d.get("healthy_mean", "?"):.2f}  '
                         f'Z={d.get("z_score", 0):+.1f}  {d["direction"]}  {d["magnitude"]}')
            lines.append(f'              Role: {d["functional_role"]}')
        lines.append('')

    # Cell type deviations
    if enriched['cell_type_deviations']:
        lines.append('── CELL TYPE COMPOSITION DEVIATIONS ────────────────────────────────────────')
        for d in enriched['cell_type_deviations']:
            lines.append(f'  {d["cell_type"]:40s}  Z={d.get("z_score", 0):+.1f}  {d.get("direction", "")}  {d.get("magnitude", "")}')
            lines.append(f'    Role: {d["functional_role"]}')
        lines.append('')

    # Pathway deviations
    if enriched['pathway_deviations']:
        lines.append('── PATHWAY DEVIATIONS ──────────────────────────────────────────────────────')
        for pw in enriched['pathway_deviations']:
            delta_str = ''
            if pw.get('deviation_from_baseline') is not None:
                delta_str = f'  Δ from healthy baseline = {pw["deviation_from_baseline"]:+.2f} log1p'
            lines.append(f'  {pw["pathway"]:40s}  {pw["direction"]}  '
                         f'active_genes={pw["n_active_genes"]}/{pw["n_total_genes"]}{delta_str}')
            if pw.get('active_genes'):
                lines.append(f'    Active: {", ".join(pw["active_genes"][:8])}')
        lines.append('')

    # Fetal reactivation
    fetal = enriched.get('fetal_reactivation')
    if fetal:
        lines.append('── FETAL / DEVELOPMENTAL REACTIVATION ──────────────────────────────────────')
        lines.append(f'  Score: {fetal["score"]:.3f}  (0=adult-like, 1=fetal-like)')
        lines.append(f'  {fetal["n_fetal_genes_reactivated"]} / {fetal["n_fetal_genes_tested"]} fetal-enriched genes above adult healthy mean')
        if fetal.get('top_reactivated_genes'):
            lines.append(f'  Top reactivated: {", ".join(fetal["top_reactivated_genes"][:8])}')
        lines.append(f'  Biological meaning: {fetal.get("biological_meaning", "")}')
        lines.append('')

    lines.append('─' * 78)
    lines.append('')
    lines.append('Please reason through this evidence step by step:')
    lines.append('')
    lines.append('STEP 1 — EVIDENCE REVIEW: What is the data actually showing? Which findings are')
    lines.append('most striking and why? Are any findings contradictory or surprising?')
    lines.append('')
    lines.append('STEP 2 — PATTERN RECOGNITION: Which findings cluster into coherent biological')
    lines.append('processes? How do the gene, cell type, and pathway findings reinforce or')
    lines.append('contradict each other?')
    lines.append('')
    lines.append('STEP 3 — MECHANISM REASONING: What biological mechanisms could produce this')
    lines.append('pattern? Walk through the molecular logic. Do not name diseases — describe')
    lines.append('the biology of what appears to be happening.')
    lines.append('')
    lines.append('STEP 4 — FUNCTIONAL IMPLICATIONS: What does this pattern mean for respiratory')
    lines.append('physiology? What functions are likely compromised? What compensatory responses')
    lines.append('might be underway?')
    lines.append('')
    lines.append('STEP 5 — UNCERTAINTY AND GAPS: What cannot be determined from this data?')
    lines.append('What additional measurements would most help clarify the picture?')
    lines.append('')
    lines.append('FINAL INTERPRETATION: A concise (3–5 sentence) biological summary suitable')
    lines.append('for a qualified respiratory researcher or clinician. Include confidence')
    lines.append('where appropriate. No disease names.')

    return '\n'.join(lines)


print('Prompt builder defined.')

In [ ]:
MODEL_ID = 'claude-opus-4-7'   # most capable — biological reasoning requires depth

SAFETY_DISCLAIMER = (
    'RESEARCH USE ONLY. This interpretation was generated by an AI reasoning engine '
    'from gene expression deviation data. It characterises biological patterns and '
    'does not constitute a clinical diagnosis. All findings require validation by '
    'certified medical professionals using approved diagnostic procedures. '
    'Do not use this output for clinical decision-making without expert review.'
)


def reason(report: dict) -> dict:
    """
    Full reasoning pipeline:
      report (DeviationReport dict)
        → triage → cluster → enrich → prompt → Claude → BiologicalInterpretation
    """
    triaged  = triage(report)
    clusters = cluster_findings(triaged)
    enriched = enrich_with_context(clusters, triaged)
    prompt   = build_reasoning_prompt(enriched)

    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=2048,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': prompt}],
    )

    reasoning_text = response.content[0].text

    return {
        'sample_id':        triaged['sample_id'],
        'overall_score':    triaged['overall_score'],
        'clusters':         clusters,
        'reasoning':        reasoning_text,
        'prompt_tokens':    response.usage.input_tokens,
        'response_tokens':  response.usage.output_tokens,
        'model':            MODEL_ID,
        'reasoned_at':      datetime.now(timezone.utc).isoformat(),
        'safety_disclaimer': SAFETY_DISCLAIMER,
    }


def print_interpretation(result: dict) -> None:
    width = 80
    print('=' * width)
    print(f'  BIOLOGICAL INTERPRETATION — {result["sample_id"]}')
    print(f'  Overall deviation score: {result["overall_score"]:.3f}')
    print(f'  Reasoning model: {result["model"]}')
    print(f'  Tokens: {result["prompt_tokens"]} in / {result["response_tokens"]} out')
    print('=' * width)

    if result['clusters']:
        print('\nBIOLOGICAL PROCESSES DETECTED:')
        for cl in result['clusters']:
            print(f'  • {cl["process"]}  ({cl["evidence_count"]} findings)')

    print('\nCHAIN-OF-THOUGHT REASONING:')
    print('-' * width)
    print(result['reasoning'])

    print('\n' + '-' * width)
    print('SAFETY DISCLAIMER:')
    print(textwrap.fill(result['safety_disclaimer'], width=width))
    print('=' * width)


print('Reasoning engine defined. Ready to run examples.')

---
## Section 7 — Example 1: Perfect Healthy Control

A sample with every gene at the healthy mean and all cell type fractions at baseline.
The reasoning engine should return a low-score report with no significant biological processes detected.

In [ ]:
healthy_report = {
    'sample_id':                'healthy_control_01',
    'overall_deviation_score':  0.04,
    'healthy_reference_cells':  meta.get('n_cells', 1_200_000),
    'healthy_reference_donors': meta.get('n_donors', 107),
    'model_built_at':           meta.get('built_at', '2025-01-01'),
    'cell_type_deviations':     [],   # no composition deviations
    'gene_deviations':          [],   # all genes within ±2σ
    'pathway_deviations':       [],   # all pathways at healthy baseline
    'fetal_reactivation': {
        'score':                    0.02,
        'n_fetal_genes_tested':     412,
        'n_fetal_genes_reactivated': 3,
        'top_reactivated_genes':    [],
        'interpretation':           'Adult-like fetal gene programme. No developmental reactivation.',
    },
    'summary': 'Sample is indistinguishable from the healthy reference.',
}

result_healthy = reason(healthy_report)
print_interpretation(result_healthy)

---
## Section 8 — Example 2: Fibrotic Lung Pattern

Elevated TGF-β pathway, increased collagen genes (COL1A1, COL3A1, FN1),
myofibroblast expansion (ACTA2+), AT2 depletion, suppressed surfactant.
The engine should identify active extracellular matrix remodelling, alveolar damage,
and the logic of impaired gas exchange through alveolar architectural disruption.

In [ ]:
fibrotic_report = {
    'sample_id':                'fibrotic_pattern_01',
    'overall_deviation_score':  0.71,
    'healthy_reference_cells':  meta.get('n_cells', 1_200_000),
    'healthy_reference_donors': meta.get('n_donors', 107),
    'model_built_at':           meta.get('built_at', '2025-01-01'),
    'cell_type_deviations': [
        {'cell_type': 'AT2', 'compartment': 'alveolar', 'estimated_fraction': 0.021,
         'healthy_mean_fraction': 0.069, 'healthy_std_fraction': 0.018,
         'z_score': -2.67, 'direction': 'depleted', 'magnitude': 'severe',
         'interpretation': 'AT2 cells severely depleted — surfactant production and alveolar repair capacity reduced'},
        {'cell_type': 'fibroblast', 'compartment': 'stromal', 'estimated_fraction': 0.089,
         'healthy_mean_fraction': 0.031, 'healthy_std_fraction': 0.012,
         'z_score': +4.83, 'direction': 'expanded', 'magnitude': 'severe',
         'interpretation': 'Fibroblast compartment severely expanded — hallmark of active stromal remodelling'},
        {'cell_type': 'alveolar macrophage', 'compartment': 'immune', 'estimated_fraction': 0.062,
         'healthy_mean_fraction': 0.121, 'healthy_std_fraction': 0.029,
         'z_score': -2.03, 'direction': 'depleted', 'magnitude': 'moderate',
         'interpretation': 'Alveolar macrophage depletion may reduce innate defence capacity'},
    ],
    'gene_deviations': [
        {'gene': 'COL1A1', 'sample_value': 3.81, 'healthy_mean': 0.29, 'healthy_std': 0.22,
         'healthy_p5': 0.0, 'healthy_p95': 0.71, 'z_score': +16.0, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'COL3A1', 'sample_value': 3.44, 'healthy_mean': 0.33, 'healthy_std': 0.24,
         'healthy_p5': 0.0, 'healthy_p95': 0.79, 'z_score': +13.0, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'TGFB1',  'sample_value': 2.91, 'healthy_mean': 0.41, 'healthy_std': 0.28,
         'healthy_p5': 0.0, 'healthy_p95': 0.94, 'z_score': +8.9, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'ACTA2',  'sample_value': 2.74, 'healthy_mean': 0.38, 'healthy_std': 0.27,
         'healthy_p5': 0.0, 'healthy_p95': 0.87, 'z_score': +8.7, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'FN1',    'sample_value': 2.62, 'healthy_mean': 0.52, 'healthy_std': 0.31,
         'healthy_p5': 0.0, 'healthy_p95': 1.09, 'z_score': +6.8, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'SFTPC',  'sample_value': 0.44, 'healthy_mean': 3.82, 'healthy_std': 1.91,
         'healthy_p5': 0.12, 'healthy_p95': 7.71, 'z_score': -1.77, 'direction': 'suppressed', 'magnitude': 'mild'},
        {'gene': 'SFTPB',  'sample_value': 0.38, 'healthy_mean': 3.14, 'healthy_std': 1.77,
         'healthy_p5': 0.09, 'healthy_p95': 6.44, 'z_score': -1.56, 'direction': 'suppressed', 'magnitude': 'mild'},
        {'gene': 'SPP1',   'sample_value': 3.12, 'healthy_mean': 1.83, 'healthy_std': 1.09,
         'healthy_p5': 0.31, 'healthy_p95': 3.91, 'z_score': +1.18, 'direction': 'elevated', 'magnitude': 'mild'},
    ],
    'pathway_deviations': [
        {'pathway': 'TGF-β / Fibrosis', 'category': 'fibrosis_ecm', 'direction': 'over-active',
         'n_active_genes': 7, 'n_total_genes': 8, 'avg_expression': 3.12,
         'trigger_baseline_expr': 0.38, 'deviation_from_baseline': +2.74,
         'active_genes': ['TGFB1', 'COL1A1', 'COL3A1', 'ACTA2', 'FN1', 'MMP2', 'TIMP1'],
         'interpretation': 'TGF-β pathway severely over-active — driving ECM deposition and myofibroblast differentiation'},
        {'pathway': 'Surfactant / AT2 Function', 'category': 'alveolar', 'direction': 'suppressed',
         'n_active_genes': 2, 'n_total_genes': 6, 'avg_expression': 0.41,
         'trigger_baseline_expr': 2.87, 'deviation_from_baseline': -2.46,
         'active_genes': ['ABCA3', 'NKX2-1'],
         'interpretation': 'Surfactant system severely suppressed — AT2 cell loss or dysfunction'},
        {'pathway': 'Hypoxia / HIF Response', 'category': 'metabolic_stress', 'direction': 'over-active',
         'n_active_genes': 4, 'n_total_genes': 7, 'avg_expression': 1.44,
         'trigger_baseline_expr': 0.61, 'deviation_from_baseline': +0.83,
         'active_genes': ['HIF1A', 'VEGFA', 'LDHA', 'SLC2A1'],
         'interpretation': 'Mild hypoxic stress response — may reflect impaired gas exchange'},
    ],
    'fetal_reactivation': {
        'score': 0.08,
        'n_fetal_genes_tested': 412,
        'n_fetal_genes_reactivated': 31,
        'top_reactivated_genes': ['COL3A1', 'FN1', 'VIM'],
        'interpretation': 'Low-level fetal programme reactivation — embryonic ECM genes re-expressed, consistent with injury-repair state',
    },
    'summary': 'Severe ECM remodelling pattern with AT2 depletion and fibroblast expansion.',
}

result_fibrotic = reason(fibrotic_report)
print_interpretation(result_fibrotic)

---
## Section 9 — Example 3: Viral / Interferon Storm

Massive interferon-stimulated gene induction (MX1, ISG15, IFIT1, OAS1),
cytokine amplification (IL6, CXCL10, CXCL9), CD8+ T cell and NK cell expansion,
alveolar macrophage activation (SPP1+). No structural remodelling — the acute
inflammation is the primary signal.

In [ ]:
viral_report = {
    'sample_id':                'viral_ifn_storm_01',
    'overall_deviation_score':  0.62,
    'healthy_reference_cells':  meta.get('n_cells', 1_200_000),
    'healthy_reference_donors': meta.get('n_donors', 107),
    'model_built_at':           meta.get('built_at', '2025-01-01'),
    'cell_type_deviations': [
        {'cell_type': 'CD8+ T cell', 'compartment': 'immune', 'estimated_fraction': 0.091,
         'healthy_mean_fraction': 0.050, 'healthy_std_fraction': 0.016,
         'z_score': +2.56, 'direction': 'expanded', 'magnitude': 'moderate',
         'interpretation': 'Cytotoxic T cell expansion — active adaptive immune response'},
        {'cell_type': 'NK cell', 'compartment': 'immune', 'estimated_fraction': 0.044,
         'healthy_mean_fraction': 0.018, 'healthy_std_fraction': 0.009,
         'z_score': +2.89, 'direction': 'expanded', 'magnitude': 'moderate',
         'interpretation': 'NK cell expansion — innate cytotoxic response'},
        {'cell_type': 'alveolar macrophage', 'compartment': 'immune', 'estimated_fraction': 0.184,
         'healthy_mean_fraction': 0.121, 'healthy_std_fraction': 0.029,
         'z_score': +2.17, 'direction': 'expanded', 'magnitude': 'moderate',
         'interpretation': 'Alveolar macrophage expansion — phagocytic clearance and cytokine production'},
    ],
    'gene_deviations': [
        {'gene': 'MX1',   'sample_value': 4.81, 'healthy_mean': 0.11, 'healthy_std': 0.09,
         'healthy_p5': 0.0, 'healthy_p95': 0.28, 'z_score': +52.2, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'ISG15', 'sample_value': 4.22, 'healthy_mean': 0.18, 'healthy_std': 0.14,
         'healthy_p5': 0.0, 'healthy_p95': 0.44, 'z_score': +28.9, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'IFIT1', 'sample_value': 3.91, 'healthy_mean': 0.09, 'healthy_std': 0.07,
         'healthy_p5': 0.0, 'healthy_p95': 0.22, 'z_score': +54.6, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'OAS1',  'sample_value': 3.44, 'healthy_mean': 0.16, 'healthy_std': 0.11,
         'healthy_p5': 0.0, 'healthy_p95': 0.38, 'z_score': +29.8, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'STAT1', 'sample_value': 3.21, 'healthy_mean': 0.22, 'healthy_std': 0.15,
         'healthy_p5': 0.03, 'healthy_p95': 0.51, 'z_score': +19.9, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'IL6',   'sample_value': 2.14, 'healthy_mean': 0.08, 'healthy_std': 0.06,
         'healthy_p5': 0.0, 'healthy_p95': 0.19, 'z_score': +34.3, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'CXCL10','sample_value': 3.88, 'healthy_mean': 0.14, 'healthy_std': 0.11,
         'healthy_p5': 0.0, 'healthy_p95': 0.34, 'z_score': +33.9, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'SPP1',  'sample_value': 3.77, 'healthy_mean': 1.83, 'healthy_std': 1.09,
         'healthy_p5': 0.31, 'healthy_p95': 3.91, 'z_score': +1.78, 'direction': 'elevated', 'magnitude': 'mild'},
    ],
    'pathway_deviations': [
        {'pathway': 'Interferon Response', 'category': 'immune_activation', 'direction': 'over-active',
         'n_active_genes': 9, 'n_total_genes': 9, 'avg_expression': 3.84,
         'trigger_baseline_expr': 0.14, 'deviation_from_baseline': +3.70,
         'active_genes': ['MX1', 'ISG15', 'IFIT1', 'IFIT3', 'OAS1', 'OAS2', 'RSAD2', 'STAT1', 'IRF7'],
         'interpretation': 'Complete interferon pathway activation — maximal antiviral gene programme'},
        {'pathway': 'NF-kB / Cytokine Storm', 'category': 'immune_activation', 'direction': 'over-active',
         'n_active_genes': 5, 'n_total_genes': 7, 'avg_expression': 2.44,
         'trigger_baseline_expr': 0.21, 'deviation_from_baseline': +2.23,
         'active_genes': ['IL6', 'IL1B', 'TNF', 'CXCL10', 'CXCL9'],
         'interpretation': 'NF-kB mediated cytokine amplification — pro-inflammatory cascade'},
        {'pathway': 'Myeloid Activation', 'category': 'immune_activation', 'direction': 'over-active',
         'n_active_genes': 4, 'n_total_genes': 6, 'avg_expression': 2.81,
         'trigger_baseline_expr': 1.44, 'deviation_from_baseline': +1.37,
         'active_genes': ['SPP1', 'MARCO', 'CD68', 'APOE'],
         'interpretation': 'Myeloid activation — macrophage-driven inflammatory amplification'},
    ],
    'fetal_reactivation': {
        'score': 0.03,
        'n_fetal_genes_tested': 412,
        'n_fetal_genes_reactivated': 11,
        'top_reactivated_genes': [],
        'interpretation': 'Minimal fetal programme reactivation — inflammation is the dominant process',
    },
    'summary': 'Massive interferon storm with NF-kB cytokine cascade and cytotoxic immune expansion.',
}

result_viral = reason(viral_report)
print_interpretation(result_viral)

---
## Section 10 — Example 4: Oncogenic Reprogramming

Active cell cycle (MKI67, TOP2A, PCNA), loss of tumour suppressors (TP53, RB1),
fetal programme reactivation (high score), no strong inflammatory signal.
The engine should identify uncontrolled proliferation and developmental dedifferentiation
without naming the condition.

In [ ]:
oncogenic_report = {
    'sample_id':                'oncogenic_pattern_01',
    'overall_deviation_score':  0.78,
    'healthy_reference_cells':  meta.get('n_cells', 1_200_000),
    'healthy_reference_donors': meta.get('n_donors', 107),
    'model_built_at':           meta.get('built_at', '2025-01-01'),
    'cell_type_deviations': [
        {'cell_type': 'basal cell', 'compartment': 'airway', 'estimated_fraction': 0.241,
         'healthy_mean_fraction': 0.052, 'healthy_std_fraction': 0.018,
         'z_score': +10.5, 'direction': 'expanded', 'magnitude': 'severe',
         'interpretation': 'Massive expansion of basal progenitor compartment — aberrant stem cell activity'},
        {'cell_type': 'multiciliated columnar cell', 'compartment': 'airway', 'estimated_fraction': 0.011,
         'healthy_mean_fraction': 0.049, 'healthy_std_fraction': 0.013,
         'z_score': -2.92, 'direction': 'depleted', 'magnitude': 'moderate',
         'interpretation': 'Differentiated ciliated cells lost — failure of terminal differentiation'},
        {'cell_type': 'AT2', 'compartment': 'alveolar', 'estimated_fraction': 0.031,
         'healthy_mean_fraction': 0.069, 'healthy_std_fraction': 0.018,
         'z_score': -2.11, 'direction': 'depleted', 'magnitude': 'moderate',
         'interpretation': 'AT2 depletion may reflect replacement by proliferating undifferentiated cells'},
    ],
    'gene_deviations': [
        {'gene': 'MKI67', 'sample_value': 4.22, 'healthy_mean': 0.04, 'healthy_std': 0.04,
         'healthy_p5': 0.0, 'healthy_p95': 0.11, 'z_score': +104.5, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'TOP2A', 'sample_value': 3.91, 'healthy_mean': 0.03, 'healthy_std': 0.03,
         'healthy_p5': 0.0, 'healthy_p95': 0.09, 'z_score': +129.7, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'PCNA',  'sample_value': 3.44, 'healthy_mean': 0.21, 'healthy_std': 0.17,
         'healthy_p5': 0.0, 'healthy_p95': 0.54, 'z_score': +18.9, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'CCND1', 'sample_value': 2.88, 'healthy_mean': 0.31, 'healthy_std': 0.22,
         'healthy_p5': 0.0, 'healthy_p95': 0.74, 'z_score': +11.7, 'direction': 'elevated', 'magnitude': 'severe'},
        {'gene': 'TP53',  'sample_value': 0.09, 'healthy_mean': 0.62, 'healthy_std': 0.38,
         'healthy_p5': 0.04, 'healthy_p95': 1.34, 'z_score': -1.39, 'direction': 'suppressed', 'magnitude': 'mild'},
        {'gene': 'RB1',   'sample_value': 0.11, 'healthy_mean': 0.74, 'healthy_std': 0.41,
         'healthy_p5': 0.06, 'healthy_p95': 1.56, 'z_score': -1.54, 'direction': 'suppressed', 'magnitude': 'mild'},
        {'gene': 'CDKN2A','sample_value': 0.07, 'healthy_mean': 0.44, 'healthy_std': 0.29,
         'healthy_p5': 0.0, 'healthy_p95': 0.99, 'z_score': -1.28, 'direction': 'suppressed', 'magnitude': 'mild'},
        {'gene': 'NKX2-1','sample_value': 0.22, 'healthy_mean': 1.89, 'healthy_std': 1.11,
         'healthy_p5': 0.08, 'healthy_p95': 4.01, 'z_score': -1.50, 'direction': 'suppressed', 'magnitude': 'mild'},
    ],
    'pathway_deviations': [
        {'pathway': 'Cell Proliferation / Oncogenic', 'category': 'oncogenic', 'direction': 'over-active',
         'n_active_genes': 6, 'n_total_genes': 7, 'avg_expression': 3.24,
         'trigger_baseline_expr': 0.09, 'deviation_from_baseline': +3.15,
         'active_genes': ['MKI67', 'TOP2A', 'PCNA', 'CCND1', 'CCNE1', 'AURKB'],
         'interpretation': 'Cell cycle massively over-activated — uncontrolled proliferation'},
        {'pathway': 'Surfactant / AT2 Function', 'category': 'alveolar', 'direction': 'suppressed',
         'n_active_genes': 1, 'n_total_genes': 6, 'avg_expression': 0.31,
         'trigger_baseline_expr': 2.87, 'deviation_from_baseline': -2.56,
         'active_genes': ['ABCA3'],
         'interpretation': 'Surfactant programme nearly silent — AT2 identity largely lost'},
    ],
    'fetal_reactivation': {
        'score': 0.54,
        'n_fetal_genes_tested': 412,
        'n_fetal_genes_reactivated': 218,
        'top_reactivated_genes': ['SOX2', 'NANOG', 'HMGA2', 'LIN28A', 'IGF2BP1', 'TWIST1', 'CDH2'],
        'interpretation': 'High fetal reactivation score — widespread re-expression of developmental programmes; hallmark of dedifferentiation toward an immature cell state',
    },
    'summary': 'Massive proliferation signal with loss of differentiation markers and high fetal reactivation.',
}

result_oncogenic = reason(oncogenic_report)
print_interpretation(result_oncogenic)

---
## Section 11 — Side-by-Side Comparison

Compare all four examples: overall score, processes detected, and the arc of biological reasoning.

In [ ]:
all_results = [
    ('Healthy control',          result_healthy),
    ('Fibrotic pattern',         result_fibrotic),
    ('Viral / interferon storm', result_viral),
    ('Oncogenic reprogramming',  result_oncogenic),
]

print(f'{'Sample':<30} {'Score':>6}  Biological processes detected')
print('-' * 90)
for label, result in all_results:
    score    = result['overall_score']
    clusters = result['clusters']
    procs    = ', '.join(cl['process'] for cl in clusters) if clusters else 'None (within healthy range)'
    print(f'{label:<30} {score:>6.3f}  {procs}')

print()
print('API usage (tokens in / out):')
for label, result in all_results:
    print(f'  {label:<30}  {result["prompt_tokens"]:>5} in  {result["response_tokens"]:>5} out')

---
## Section 12 — Integration Path

How this reasoning engine connects to the FastAPI backend (Layer 2).

In [ ]:
print("""
INTEGRATION ARCHITECTURE
========================

Current flow (Notebook 03):
  Gene expression sample
    → anomaly_detector.analyse()       (Python, FastAPI service)
    → DeviationReport                  (Pydantic schema)

Extended flow (this notebook → app/services/reasoning_engine.py):
  DeviationReport
    → triage()
    → cluster_findings()
    → enrich_with_context()
    → build_reasoning_prompt()
    → Claude API (claude-opus-4-7)
    → BiologicalInterpretation         (new Pydantic schema)

New API endpoint to add:
  POST /api/v1/interpret
    Input:  DeviationReport (or sample_id if already cached)
    Output: BiologicalInterpretation
      - sample_id
      - clusters: list of biological process clusters with evidence
      - reasoning: full chain-of-thought text
      - reasoned_at
      - model
      - safety_disclaimer

New Pydantic schemas to add to app/models/schemas.py:
  BiologicalCluster
    - process: str
    - evidence_count: int
    - gene_evidence: list[dict]
    - ct_evidence: list[dict]
    - pathway_evidence: list[dict]

  BiologicalInterpretation
    - sample_id: str
    - overall_score: float
    - clusters: list[BiologicalCluster]
    - reasoning: str
    - reasoned_at: datetime
    - model: str
    - safety_disclaimer: str

Cost estimate (claude-opus-4-7):
  ~1,500 tokens input + ~2,000 tokens output per sample
  = roughly $0.05–0.10 per interpretation at current pricing
  Cache the system prompt (it never changes) to reduce input cost by ~60%.
""")

---
## Section 13 — What Comes Next

After this notebook is validated in Colab:

1. **Port to `app/services/reasoning_engine.py`** — move `triage()`, `cluster_findings()`,
   `enrich_with_context()`, `build_reasoning_prompt()`, and `reason()` into the FastAPI service.
   Add `ANTHROPIC_API_KEY` to environment config.

2. **Add `POST /api/v1/interpret`** — new endpoint in `app/api/v1/model.py`.
   Input is a `DeviationReport` dict; output is `BiologicalInterpretation`.

3. **Add new Pydantic schemas** — `BiologicalCluster` and `BiologicalInterpretation`
   in `app/models/schemas.py`.

4. **Cache the system prompt** — use Anthropic prompt caching on the system prompt
   (it never changes per request) to reduce API cost significantly.

5. **Notebook 05** — Expert Network integration: take a `BiologicalInterpretation`,
   let an expert annotate it, and pipe that annotation back as training signal.

---
## Section 14 — System Prompt (Inspect & Edit)

The system prompt is the fixed instruction Claude receives before any sample data.
Every interpretation passes through it. Editing it changes the reasoning style,
level of detail, constraint strictness, and output format.

**To iterate**: edit `ACTIVE_SYSTEM_PROMPT` in the next cell, re-run the cell,
then re-run any of the example cells (Sections 7–10). The `test()` function in
Section 15 always uses `ACTIVE_SYSTEM_PROMPT`.

The annotated breakdown below explains what each block does and why —
so you know which part to edit when a specific behaviour is wrong.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ACTIVE_SYSTEM_PROMPT — edit here, re-run, then re-run test() calls.
# Version 6: (1) Part B nodes labelled [MEASURED]/[INFERRED]; (2) Step 4 adds
# Physiological Priority Ranking (Tier 1–5 by immediacy of threat).
# ─────────────────────────────────────────────────────────────────────────────

ACTIVE_SYSTEM_PROMPT = """\
You are a respiratory biology research assistant integrated into the HEALTH platform.

Your role is to interpret gene expression deviation data produced by the Healthy Respiratory
Model — a reference built from 1.2 million healthy human lung cells (Human Lung Cell Atlas).

━━━ SAFETY BOUNDARY ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
You MUST NOT name specific clinical conditions, syndromes, or diagnoses.
You MAY name biological mechanisms, molecular pathways, and cellular processes using
precise scientific terminology, even when those mechanisms are associated with known
clinical conditions.
The test: if your statement could be placed in a patient's medical record as a clinical
finding, it has crossed the line. Mechanism descriptions belong in a research report.
Diagnoses belong to a physician reviewing the full clinical picture.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ADDITIONAL CONSTRAINTS:
- Do not speculate beyond the biological evidence provided. You may note what additional
  evidence would be expected if a hypothesis is correct, but distinguish this clearly
  from confirmed findings.
- When findings are ambiguous or contradictory, explain both interpretations.

Your output characterises BIOLOGY. You describe mechanisms. Clinicians draw conclusions.

━━━ CONFIDENCE GRADING (required in STEP 2) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  HIGH     — ≥ 3 independent dimensions converge (gene + cell type + pathway)
  MODERATE — 2 dimensions, or 1 with |Z| ≥ 8 or |Δ| ≥ 2.5
  LOW      — single data point or one dimension only
Cite at least one Z-score or pathway delta per claim in the FINAL INTERPRETATION.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━ HEALTHY SAMPLE HANDLING ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Score < 0.10 with no finding ≥ |Z| 2.0 or |Δ| 0.8: respond in three short paragraphs.
Do not generate reasoning steps when there is no evidence to reason about.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

For samples with score ≥ 0.10 or at least one significant finding, work through
each step in order. Do not skip or merge steps.

STEP 1 — EVIDENCE REVIEW
Identify the most striking findings. Cite specific Z-scores and pathway deltas.
Flag anything contradictory or surprising. Note whether findings span multiple
dimensions or concentrate in one.

STEP 2 — PATTERN RECOGNITION
Group findings into coherent biological processes. For each:
  Confidence: HIGH / MODERATE / LOW
  Evidence: list per dimension (gene, cell type, pathway)
  Coherence: where dimensions reinforce or conflict

STEP 2B — CROSS-VALIDATION
For every finding identified in Steps 1 and 2, classify it as:
  PREDICTED  — the expected downstream consequence of a confirmed process from Step 2.
               It corroborates but adds no independent evidence.
  NEUTRAL    — unrelated to any detected process; present but unexplained.
  ANOMALOUS  — contradicts what the confirmed processes would predict.
               This is the highest-priority category: an anomalous finding either
               reveals a second process not yet identified, or exposes a flaw in
               the Step 2 interpretation.
Present results as a labelled list. Then:
  - If ANOMALOUS findings exist: state what they contradict and what alternative
    process could explain them. These become the focus of Step 3.
  - If NO ANOMALOUS findings exist: state this explicitly. A fully coherent pattern
    (all findings PREDICTED or NEUTRAL) increases confidence in the Step 2
    interpretation — note this, but also ask: is the absence of anomaly because the
    pattern is genuinely coherent, or because the current gene panel cannot detect
    what would be anomalous?

STEP 3 — MECHANISM, CAUSAL ORDERING, AND STAGE

Part A — Molecular inventory
List signalling nodes as brief chains (one per line, not prose):
  Receptor → adaptor → transcription factor → confirmed target genes
If anomalous findings exist from Step 2B, include the mechanism that would explain
them as a separate chain marked [ANOMALY EXPLANATION: speculative].

Part B — Causal ordering
Arrange the inventory into a single A → B → C sequence.
Label each NODE:  [MEASURED] = directly observed in the data (has a Z-score or Δ);
                  [INFERRED] = mechanistic background knowledge, not directly measured.
Label each ARROW: DIRECT / INFERRED / UNKNOWN.
A chain of [MEASURED]→[MEASURED] nodes is data-grounded.
A chain that passes through [INFERRED] nodes is mechanistically plausible but not confirmed.
Identify: primary perturbation, downstream consequences, self-amplifying loops.
If the primary perturbation is unresolvable, state this and name what would resolve it.

Part C — Stage inference
Assign: INITIATING / ACTIVE / ESTABLISHED / RESOLVING.
State which stage fits and the single finding that, if different, would shift it.

STEP 4 — FUNCTIONAL IMPLICATIONS
Use the causal chain (Part B) and stage (Part C) to trace specific respiratory
physiology consequences node by node. Assess reversibility at each node given the
stage. Note compensatory responses visible in the data.

After the node-by-node analysis, add a PHYSIOLOGICAL PRIORITY RANKING.
Rank each consequence from most to least immediate threat to core respiratory
function using this hierarchy (Tier 1 = highest, most immediate threat):
  Tier 1 — Gas exchange (alveolar O₂/CO₂ diffusion; surfactant integrity)
  Tier 2 — Ventilation mechanics (airway patency, compliance, mucus clearance)
  Tier 3 — Host defence (innate + adaptive barriers against pathogen entry/spread)
  Tier 4 — Structural integrity (ECM architecture, alveolar wall geometry)
  Tier 5 — Metabolic homeostasis (cell-level energy, redox, proteostasis)
For each ranked item state: which finding drives it, its stage-informed
reversibility (HIGH / MODERATE / LOW / UNKNOWN), and whether a compensatory
response is already visible in the data. Omit tiers with no relevant findings.

STEP 5 — UNCERTAINTY AND GAPS
Name the single measurement that would most increase confidence in the causal ordering
and stage. Explain specifically what it would reveal and how it would change the
interpretation — including what it would do to the anomaly explanation if one exists.

FINAL INTERPRETATION
3–5 sentences covering: overall confidence, active processes with cited Z-scores/deltas,
stage inference, any anomalous findings and their implication, and the leading
uncertainty. End with the single most important biological question this profile raises.
No clinical condition names.
"""

SYSTEM_PROMPT = ACTIVE_SYSTEM_PROMPT
print('ACTIVE_SYSTEM_PROMPT (v6) loaded.')
print(f'Length: {len(ACTIVE_SYSTEM_PROMPT):,} chars')
print()
print('New in v6:')
print('  + Part B: every NODE labelled [MEASURED] or [INFERRED]')
print('  + Part B: [MEASURED]→[MEASURED] chains distinguished from inferred pathways')
print('  + Step 4: PHYSIOLOGICAL PRIORITY RANKING (Tier 1–5) added after node analysis')
print('  + Tier hierarchy: gas exchange → ventilation → host defence → structure → metabolism')
print()
print('Retained from v5: healthy branch, confidence grading, Part A/B/C, stage inference,')
print('  STEP 2B cross-validation, ANOMALOUS/PREDICTED/NEUTRAL, single-measurement Step 5.')


---
## Section 15 — Interactive Testing

Two ways to test:

**Quick call** — `test()` accepts plain dicts of gene→z_score, cell_type→z_score,
pathway→delta. It builds the full DeviationReport structure, runs the pipeline,
and prints the interpretation. Good for rapid iteration.

**Full control** — build a complete `report` dict (same schema as DeviationReport)
and pass it directly to `reason()`. Use this when you want to test a specific
combination of values that the quick form doesn't express precisely.

In [ ]:
# ── Helper: classify deviation magnitude from |Z| ─────────────────────────
def _magnitude(z: float) -> str:
    az = abs(z)
    if az >= 4.0:  return 'severe'
    if az >= 2.5:  return 'moderate'
    return 'mild'


def gene_dev(gene: str, z_score: float, sample_value: float | None = None) -> dict:
    """
    Build a gene deviation entry from just gene name + Z-score.
    Looks up healthy_mean from the loaded gene_stats artefact.
    sample_value defaults to healthy_mean + z_score * healthy_std if not given.
    """
    stats = gene_stats.get(gene, {})
    h_mean = stats.get('mean', 1.0)
    h_std  = stats.get('std',  0.5)
    sv     = sample_value if sample_value is not None else max(0.0, h_mean + z_score * h_std)
    return {
        'gene':          gene,
        'sample_value':  round(sv, 3),
        'healthy_mean':  round(h_mean, 3),
        'healthy_std':   round(h_std, 3),
        'healthy_p5':    stats.get('p5',  0.0),
        'healthy_p95':   stats.get('p95', h_mean * 2),
        'z_score':       round(z_score, 2),
        'direction':     'elevated' if z_score > 0 else 'suppressed',
        'magnitude':     _magnitude(z_score),
    }


# Compartment inference from cell type name
_CT_COMPARTMENTS = {
    'alveolar':   ['AT1', 'AT2', 'alveolar'],
    'airway':     ['basal', 'ciliated', 'multiciliated', 'goblet', 'club', 'secretory', 'airway'],
    'immune':     ['macrophage', 'T cell', 'B cell', 'NK', 'monocyte', 'dendritic', 'mast', 'neutrophil', 'lymphoid', 'myeloid'],
    'stromal':    ['fibroblast', 'myofibroblast', 'smooth muscle', 'pericyte', 'stromal'],
    'vascular':   ['endothelial', 'vascular', 'aerocyte', 'capillary'],
}

def _infer_compartment(cell_type: str) -> str:
    ct_lower = cell_type.lower()
    for compartment, keywords in _CT_COMPARTMENTS.items():
        if any(kw in ct_lower for kw in keywords):
            return compartment
    return 'other'


def ct_dev(cell_type: str, z_score: float) -> dict:
    """
    Build a cell type deviation entry from cell type name + Z-score.
    Looks up healthy fractions from the loaded composition artefact.
    """
    comp = composition.get(cell_type, {})
    h_mean = comp.get('mean_fraction', 0.05)
    h_std  = comp.get('std_fraction',  0.015)
    est    = max(0.001, h_mean + z_score * h_std)
    return {
        'cell_type':              cell_type,
        'compartment':            _infer_compartment(cell_type),
        'estimated_fraction':     round(est, 4),
        'healthy_mean_fraction':  round(h_mean, 4),
        'healthy_std_fraction':   round(h_std, 4),
        'z_score':                round(z_score, 2),
        'direction':              'expanded' if z_score > 0 else 'depleted',
        'magnitude':              _magnitude(z_score),
        'interpretation':         '',
    }


def pathway_dev(pathway_name: str, delta: float, active_genes: list[str] | None = None) -> dict:
    """
    Build a pathway deviation entry from pathway name + delta (log1p CP10K units above/below baseline).
    Looks up baseline stats from the loaded pathway_baselines artefact.
    delta > 0  → over-active;  delta < 0 → suppressed.
    """
    baseline = pathway_baselines.get(pathway_name, {})
    baseline_mean = baseline.get('trigger_mean_expr') or 0.5
    genes_tracked = baseline.get('genes_tracked', {})
    n_total = len([g for g, v in genes_tracked.items() if v.get('role') == 'trigger']) or 6

    if active_genes is None:
        # Use known trigger genes from the pathway atlas as plausible defaults
        triggers = [g for g, v in genes_tracked.items() if v.get('role') == 'trigger']
        n_active = max(1, int(n_total * (0.8 if delta > 1 else 0.4 if delta > 0 else 0.15)))
        active_genes = triggers[:n_active]
    else:
        n_active = len(active_genes)

    return {
        'pathway':                 pathway_name,
        'category':                baseline.get('category', 'unknown'),
        'direction':               'over-active' if delta > 0 else 'suppressed',
        'n_active_genes':          n_active,
        'n_total_genes':           n_total,
        'avg_expression':          round(max(0.0, baseline_mean + delta), 3),
        'trigger_baseline_expr':   round(baseline_mean, 3),
        'deviation_from_baseline': round(delta, 3),
        'active_genes':            active_genes,
        'interpretation':          '',
    }


def fetal_dev(score: float, top_genes: list[str] | None = None) -> dict:
    """Build a fetal reactivation entry from a score (0–1)."""
    n_tested = len(fetal_reference.get('fetal_enriched', {})) or 412
    n_react  = int(n_tested * score * 0.8)  # approximate
    if top_genes is None and fetal_reference.get('fetal_enriched'):
        all_fetal = list(fetal_reference['fetal_enriched'].keys())
        top_genes = all_fetal[:int(len(all_fetal) * score)][:10]
    return {
        'score':                     round(score, 3),
        'n_fetal_genes_tested':      n_tested,
        'n_fetal_genes_reactivated': n_react,
        'top_reactivated_genes':     top_genes or [],
        'interpretation':            '',
    }


print('Helper constructors ready: gene_dev(), ct_dev(), pathway_dev(), fetal_dev()')

In [ ]:
def _overall_score(genes: list[dict], cts: list[dict], pathways: list[dict], fetal: dict | None) -> float:
    """Estimate the overall deviation score from the inputs, matching anomaly_detector logic."""
    # Gene component: fraction of genes with |Z| >= 2, scaled by severity
    if genes:
        gene_scores = [min(1.0, abs(g['z_score']) / 8.0) for g in genes]
        gene_comp   = sum(gene_scores) / (len(gene_scores) + 5)  # denominator > len to bound < 1
    else:
        gene_comp = 0.0

    # Cell type component: max |Z| normalised
    ct_comp = min(1.0, max((abs(c['z_score']) for c in cts), default=0.0) / 5.0) if cts else 0.0

    # Pathway component: mean |delta| normalised
    if pathways:
        deltas = [abs(p.get('deviation_from_baseline') or 0) for p in pathways]
        pw_comp = min(1.0, sum(deltas) / (len(deltas) * 3.0))
    else:
        pw_comp = 0.0

    # Fetal component
    fetal_comp = (fetal['score'] if fetal else 0.0)

    return round(0.35 * ct_comp + 0.30 * gene_comp + 0.25 * pw_comp + 0.10 * fetal_comp, 3)


def test(
    sample_id: str,
    *,
    genes:      dict[str, float] | None = None,   # {gene: z_score}
    cell_types: dict[str, float] | None = None,   # {cell_type: z_score}
    pathways:   dict[str, float] | None = None,   # {pathway_name: delta}
    fetal_score: float = 0.02,
    fetal_genes: list[str] | None = None,
    overall_score: float | None = None,
    verbose_prompt: bool = False,
) -> dict:
    """
    Single-call entry point for interactive testing.

    Minimal example — just gene Z-scores:
        test('my_sample', genes={'MX1': +30, 'ISG15': +20, 'SFTPC': -3})

    With cell types and pathways:
        test('my_sample',
             genes      = {'COL1A1': +10, 'TGFB1': +6},
             cell_types = {'fibroblast': +4, 'AT2': -3},
             pathways   = {'TGF-β / Fibrosis': +2.5, 'Surfactant / AT2 Function': -2.0},
             fetal_score = 0.05)

    Pathways must exactly match keys in pathway_baselines.json.
    Print available pathway names with: print(list(pathway_baselines.keys()))
    """
    gene_devs = [gene_dev(g, z) for g, z in (genes or {}).items()]
    ct_devs   = [ct_dev(ct, z)  for ct, z in (cell_types or {}).items()]
    pw_devs   = [pathway_dev(pw, d) for pw, d in (pathways or {}).items()]
    fetal     = fetal_dev(fetal_score, fetal_genes)

    score = overall_score if overall_score is not None else _overall_score(gene_devs, ct_devs, pw_devs, fetal)

    report = {
        'sample_id':                sample_id,
        'overall_deviation_score':  score,
        'healthy_reference_cells':  meta.get('n_cells', 1_200_000),
        'healthy_reference_donors': meta.get('n_donors', 107),
        'model_built_at':           meta.get('built_at', ''),
        'gene_deviations':          gene_devs,
        'cell_type_deviations':     ct_devs,
        'pathway_deviations':       pw_devs,
        'fetal_reactivation':       fetal,
        'summary':                  '',
    }

    if verbose_prompt:
        triaged  = triage(report)
        clusters = cluster_findings(triaged)
        enriched = enrich_with_context(clusters, triaged)
        prompt   = build_reasoning_prompt(enriched)
        print('── FULL PROMPT SENT TO MODEL ──────────────────────────────────────────────')
        print(prompt)
        print('──────────────────────────────────────────────────────────────────────────')
        print()

    result = reason(report)
    print_interpretation(result)
    return result


print('test() ready.')
print()
print('Available pathway names:')
for pw in sorted(pathway_baselines.keys()):
    baseline = pathway_baselines[pw].get('trigger_mean_expr')
    print(f'  "{pw}"  (healthy baseline: {baseline:.2f} log1p)' if baseline else f'  "{pw}"  (no baseline)')

### Sandbox — Edit and Run

Modify the dict below and re-run this cell. The helper constructors fill in everything
else from the artefacts — you only supply the biological signal you want to test.

**Gene Z-score guide**: ±2 = mild deviation, ±4 = moderate, ±8+ = severe. Positive = elevated, negative = suppressed.

**Pathway delta guide**: ±0.8 log1p CP10K is the detection threshold. ±2.0 is strong. ±3.5+ is extreme.

In [ ]:
# ── Edit these inputs and re-run ────────────────────────────────────────────

SAMPLE_ID = 'my_test_sample'

# Gene Z-scores — add any gene symbol the model tracks
MY_GENES = {
    # 'GENE_NAME': z_score,
    'SFTPC':  -4.0,   # surfactant gene suppressed
    'TGFB1':  +5.0,   # pro-fibrotic signal elevated
    'COL1A1': +8.0,   # collagen deposition
    'MX1':    +2.5,   # mild interferon response
}

# Cell type Z-scores — use exact names from composition artefact
# (Run: print(list(composition.keys())) to see all available names)
MY_CELL_TYPES = {
    # 'cell type name': z_score,
    'AT2':        -2.5,
    'fibroblast': +3.5,
}

# Pathway deltas — use exact names from pathway_baselines artefact
# (Run: print(list(pathway_baselines.keys())) for all names)
MY_PATHWAYS = {
    # 'Pathway Name': delta_in_log1p_units,
    'TGF-β / Fibrosis':          +2.2,
    'Surfactant / AT2 Function': -1.8,
}

# Fetal reactivation score (0.0 = fully adult-like, 1.0 = fully fetal-like)
MY_FETAL_SCORE = 0.06

# ── Run ─────────────────────────────────────────────────────────────────────
result = test(
    SAMPLE_ID,
    genes       = MY_GENES,
    cell_types  = MY_CELL_TYPES,
    pathways    = MY_PATHWAYS,
    fetal_score = MY_FETAL_SCORE,
    verbose_prompt = False,   # set True to see the full prompt sent to the model
)

---
## Section 16 — Prompt Iteration

Run the same sample through multiple system prompt variants and compare the outputs side by side.
Use this to validate that prompt changes improve quality without breaking safety constraints.

**What to iterate on**:
- Reasoning depth (5 steps vs 3 steps vs free-form)
- Output format (prose vs structured bullet points)
- Uncertainty language (explicit probability vs qualitative hedging)
- Constraint strictness (tighter no-disease boundary vs allowing mechanism names that overlap with disease names)
- Audience calibration (research scientist vs clinician vs both)

In [ ]:
def reason_with_prompt(report: dict, system_prompt: str, label: str = '') -> dict:
    """Run reasoning with a custom system prompt. Useful for A/B prompt testing."""
    triaged  = triage(report)
    clusters = cluster_findings(triaged)
    enriched = enrich_with_context(clusters, triaged)
    prompt   = build_reasoning_prompt(enriched)

    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=2048,
        system=system_prompt,
        messages=[{'role': 'user', 'content': prompt}],
    )

    return {
        'label':           label or 'custom',
        'sample_id':       triaged['sample_id'],
        'overall_score':   triaged['overall_score'],
        'clusters':        clusters,
        'reasoning':       response.content[0].text,
        'prompt_tokens':   response.usage.input_tokens,
        'response_tokens': response.usage.output_tokens,
        'model':           MODEL_ID,
        'safety_disclaimer': SAFETY_DISCLAIMER,
    }


def compare_prompts(report: dict, variants: dict[str, str]) -> None:
    """
    Run the same report through multiple system prompt variants and print
    each output sequentially for comparison.

    variants: {label: system_prompt_string}
    """
    results = {}
    for label, prompt in variants.items():
        print(f'Running variant: {label} ...')
        results[label] = reason_with_prompt(report, prompt, label)

    print()
    for label, result in results.items():
        width = 80
        print('\n' + '█' * width)
        print(f'  VARIANT: {label}')
        print(f'  Tokens: {result["prompt_tokens"]} in / {result["response_tokens"]} out')
        print('█' * width)
        print(result['reasoning'])
        print()

    return results


print('Prompt iteration tools ready: reason_with_prompt(), compare_prompts()')

In [ ]:
# ── Prompt variant library ───────────────────────────────────────────────────
# Three prompt variants to try. Edit freely. Run compare_prompts() below.

PROMPT_VARIANTS = {

    # Variant A — current production prompt (baseline)
    'A: current (5-step CoT)': ACTIVE_SYSTEM_PROMPT,

    # Variant B — concise clinician-facing output
    # Rationale to test: expert clinicians may prefer tighter output with
    # fewer reasoning steps and a more structured final summary.
    'B: concise clinician': """\
You are a respiratory biology research assistant integrated into the HEALTH platform.
Your role is to interpret gene expression deviation data from the Healthy Respiratory Model.

CRITICAL CONSTRAINTS:
- Do NOT name specific diseases or provide clinical diagnoses.
- Restrict interpretation strictly to the provided evidence.
- Flag ambiguity or contradiction explicitly.

Your audience is qualified respiratory physicians. Be precise and concise.
Use clinical terminology where appropriate but stay within the data provided.

Output format:
1. KEY FINDINGS (3–5 bullet points — most significant deviations only)
2. BIOLOGICAL MECHANISM (2–3 sentences — what process could explain the pattern)
3. FUNCTIONAL CONSEQUENCE (1–2 sentences — what does this mean for lung function)
4. OPEN QUESTIONS (bullet list — what the data cannot resolve)
5. SUMMARY (1 sentence — the core biological message for a clinician)

No introductory sentences. Start directly with KEY FINDINGS.
""",

    # Variant C — researcher-facing with explicit uncertainty quantification
    # Rationale to test: research scientists may want probability language and
    # clear separation between high-confidence and speculative inferences.
    'C: researcher + uncertainty': """\
You are a respiratory biology research assistant integrated into the HEALTH platform.
You interpret gene expression deviation data from the Healthy Respiratory Model — a reference
built from 1.2 million healthy human lung cells (Human Lung Cell Atlas).

CRITICAL CONSTRAINTS:
- Do NOT name specific diseases or clinical diagnoses.
- Distinguish clearly between high-confidence findings (multiple lines of evidence)
  and speculative inferences (single data point or ambiguous signal).
- When evidence conflicts, explain both interpretations.

Use explicit confidence language:
- HIGH CONFIDENCE: ≥3 independent lines of evidence point to the same process.
- MODERATE: 2 lines of evidence, or 1 strong line.
- LOW / SPECULATIVE: single data point, contradictory signals, or known confounds.

Think step by step. Then produce a structured output:

EVIDENCE SUMMARY
  [List the strongest findings with their Z-scores and pathway deltas]

BIOLOGICAL PROCESSES DETECTED
  [For each detected process, state: confidence level, supporting evidence, mechanism]

ALTERNATIVE EXPLANATIONS
  [What else could explain these findings? What would distinguish the alternatives?]

RECOMMENDED FOLLOW-UP ASSAYS
  [What additional measurements would most increase confidence?]

SUMMARY
  [2–3 sentences for a research scientist. Include confidence overall.]
""",

}

print('Prompt variants defined:')
for name in PROMPT_VARIANTS:
    tokens_est = len(PROMPT_VARIANTS[name].split()) * 1.3  # rough token estimate
    print(f'  {name}  (~{int(tokens_est)} tokens)')

### Run the comparison

The cell below runs all three variants on the fibrotic report from Section 8.
Change `fibrotic_report` to any other report dict to test a different case.

**Cost note**: This calls the API 3×. Each call is ~\$0.05–0.10. To limit cost
during iteration, edit `PROMPT_VARIANTS` to contain only 2 variants, or test
each variant individually with `reason_with_prompt()`.

In [ ]:
# Choose which report to use as the comparison baseline.
# Options: fibrotic_report, viral_report, oncogenic_report, healthy_report,
#          or any report dict you built with test() above.
COMPARISON_REPORT = fibrotic_report

# Choose which variants to compare (comment out any you want to skip)
ACTIVE_VARIANTS = {
    k: v for k, v in PROMPT_VARIANTS.items()
    if k in [
        'A: current (5-step CoT)',
        'B: concise clinician',
        # 'C: researcher + uncertainty',  # uncomment to include
    ]
}

comparison_results = compare_prompts(COMPARISON_REPORT, ACTIVE_VARIANTS)

---
## Section 17 — Evaluation Criteria

When reviewing outputs, check these properties before accepting a prompt variant:

| Property | What to look for | Red flag |
|---|---|---|
| **Safety boundary** | No disease names, no diagnoses | Any specific disease name in the output |
| **Biological accuracy** | Mechanistic logic is consistent (e.g. TGF-β → myofibroblast → ECM) | Steps that skip the mechanism or contradict known biology |
| **Evidence grounding** | Every claim traces to a specific Z-score or pathway delta | Statements not supported by any finding in the report |
| **Healthy control behaviour** | Score ≈0 → brief, non-alarming output | Healthy control flagged as abnormal |
| **Proportionality** | Severe deviations (|Z|≥8) are more prominent than mild (|Z|≈2) | Mild finding given equal weight as severe finding |
| **Uncertainty language** | Ambiguous findings are flagged as such | False certainty on low-evidence signals |
| **Functional implication** | Connects molecular findings to lung physiology | Output stays at gene level without physiological context |
| **Conciseness** | Final summary is ≤5 sentences | Summary longer than the reasoning itself |

**Iterating**:
1. Identify the specific property that is failing.
2. Edit the relevant block of `ACTIVE_SYSTEM_PROMPT` in Section 14.
3. Re-run Section 14 to update `SYSTEM_PROMPT`.
4. Run `test()` in Section 15 with the same inputs to compare.
5. When satisfied, copy `ACTIVE_SYSTEM_PROMPT` into `app/services/reasoning_engine.py`.